In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from scipy import stats as st
import os
import csv
from collections import Counter
from difflib import SequenceMatcher

In [3]:
def string_similarity_match(str1, str2, threshold=90):
    # Calculate similarity ratio using SequenceMatcher
    ratio = SequenceMatcher(None, str1.lower(), str2.lower()).ratio()
    similarity_percentage = ratio * 100 
    
    return similarity_percentage >= threshold

def get_best_match_index(target_string, candidate_list, threshold=90):
    if not candidate_list:
        return None
    
    best_similarity = 0.0
    best_index = None
    
    for i, candidate in enumerate(candidate_list):
        ratio = SequenceMatcher(None, str(target_string).lower(), str(candidate).lower()).ratio()
        similarity = ratio * 100
        
        if similarity > best_similarity:
            best_similarity = similarity
            best_index = i
    
    # Return index only if it meets threshold
    if best_similarity >= threshold:
        print('best sim:', best_similarity)
    return best_index if best_similarity >= threshold else None

In [4]:
def transform_uci_dict(original_dict):
    
    transformed_dict = {'message': [], 'label': []}
    
    # Process ham messages (label = 0)
    for message in original_dict['ham']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('ham')
    
    # Process spam messages (label = 1)
    for message in original_dict['spam']:
        transformed_dict['message'].append(message)
        transformed_dict['label'].append('spam')
    
    return pd.DataFrame(transformed_dict)

In [5]:
def read_file(file_path):
    with open(file_path, 'r') as file:
        content = file.readlines()
    return content

In [6]:
uci_dataset = read_file("../Dataset/UCI_SMSSpamCollection")
uci_dataset = [i[:-1] for i in uci_dataset]
uci_dataset_ham, uci_dataset_spam = [], []
for i in uci_dataset:
    if i[0]=='h':
        uci_dataset_ham.append(i[4:])
    else:
        uci_dataset_spam.append(i[5:])

uci_dataset = {'ham': uci_dataset_ham, 'spam': uci_dataset_spam}


In [7]:
uci_dataset = transform_uci_dict(uci_dataset)
uci_dataset.head()

,message,label
0,"Go until jurong point, crazy.. Available only ...",ham
1,Ok lar... Joking wif u oni...,ham
2,U dun say so early hor... U c already then say...,ham
3,"Nah I don't think he goes to usf, he lives aro...",ham
4,Even my brother is not like to speak with me. ...,ham


In [8]:
# Counter(pd.read_csv('../Dataset/URL Data/'+'uci Dataset_'+'.csv')['URL'].to_list())

In [9]:
uci_dataset['Extracted URL'] = pd.read_csv('../Dataset/URL Data/'+'uci Dataset_'+'.csv')['URL']
uci_dataset['Message Len'] = [len(i) for i in uci_dataset['message']]
uci_dataset.head()

,message,label,Extracted URL,Message Len
0,"Go until jurong point, crazy.. Available only ...",ham,NaN,111
1,Ok lar... Joking wif u oni...,ham,NaN,29
2,U dun say so early hor... U c already then say...,ham,NaN,49
3,"Nah I don't think he goes to usf, he lives aro...",ham,NaN,61
4,Even my brother is not like to speak with me. ...,ham,NaN,77


In [10]:
uci_website_analysis_data = pd.read_csv('../Dataset/URL Data/'+'uci Websites Analysis'+'.csv')
uci_website_analysis_data = uci_website_analysis_data.drop(columns=['ham', 'spam'])
uci_website_analysis_data.head()

,Unique Url,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,www.txt43.com,www.txt43.com,114,0,200,1
1,www.fullonsms.com,www.fullonsms.com,1178,0,200,1
2,www.areyouunique.co.uk,www.areyouunique.co.uk,0,0,-1,0
3,www.Ldew.com1win150ppmx3age16,www.Ldew.com1win150ppmx3age16.,0,0,-1,0
4,www.SMS.ac/u/bootydelious,www.SMS.ac,1034,0,200,1


In [11]:
string_similarity_match('http://wiseschool.com','wiseschool.com')

False

In [12]:
uci_website_analysis_data.iloc[0][0]

C:\Users\mmia43\AppData\Local\Temp\ipykernel_24896\6499989.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  uci_website_analysis_data.iloc[0][0]


'www.txt43.com'

In [13]:
# for row in uci_website_analysis_data.itertuples(index=False):
#     print(row[0])

In [14]:
import tldextract

def FQDN(Url):
    
    if type(Url) !=str:
        return ''

    url_extract_res = tldextract.extract(Url)
    fqdn = ''
    if url_extract_res.subdomain:
        fqdn = url_extract_res.subdomain + '.' + url_extract_res.domain + '.' + url_extract_res.suffix
        # fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    else:
        fqdn = url_extract_res.domain + '.' + url_extract_res.suffix
    
    return fqdn

In [15]:
extracted_urls = uci_dataset['Extracted URL'].values
extracted_urls_fqdn = [FQDN(i) for i in extracted_urls]

# Create a dictionary for O(1) lookup instead of O(n) loop
fqdn_to_index = {fqdn: idx for idx, fqdn in enumerate(uci_website_analysis_data['FQDN'])}
website_data = uci_website_analysis_data.iloc[:, 1:6].values

# Pre-allocate lists for better performance
n_urls = len(extracted_urls)
fqdn = extracted_urls_fqdn
website_size = [''] * n_urls
text_content_len = [''] * n_urls
status_code = [''] * n_urls
parked = [''] * n_urls

# Single optimized loop with dictionary lookup
for i, url in enumerate(extracted_urls):
    if url and extracted_urls_fqdn[i]:  # Check both url and fqdn exist
        matched_idx = fqdn_to_index.get(extracted_urls_fqdn[i])  # O(1) lookup
        if matched_idx is not None:
            row_data = website_data[matched_idx]
            website_size[i] = row_data[1]
            text_content_len[i] = row_data[2]
            status_code[i] = row_data[3]
            parked[i] = row_data[4]

In [16]:
uci_dataset['FQDN'] = fqdn
uci_dataset['Website Size in KB'] = website_size
uci_dataset['Website Textual Content Length'] = text_content_len
uci_dataset['Status Code'] = status_code
uci_dataset['Parked'] = parked

In [17]:
uci_dataset = uci_dataset.replace('', np.nan)
uci_dataset.head()

C:\Users\mmia43\AppData\Local\Temp\ipykernel_24896\1973594041.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  uci_dataset = uci_dataset.replace('', np.nan)


,message,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
0,"Go until jurong point, crazy.. Available only ...",ham,NaN,111,NaN,NaN,NaN,NaN,NaN
1,Ok lar... Joking wif u oni...,ham,NaN,29,NaN,NaN,NaN,NaN,NaN
2,U dun say so early hor... U c already then say...,ham,NaN,49,NaN,NaN,NaN,NaN,NaN
3,"Nah I don't think he goes to usf, he lives aro...",ham,NaN,61,NaN,NaN,NaN,NaN,NaN
4,Even my brother is not like to speak with me. ...,ham,NaN,77,NaN,NaN,NaN,NaN,NaN


In [18]:
# Counter(uci_dataset['FQDN'].to_list())
uci_dataset[(uci_dataset['Extracted URL'].notna()) & (uci_dataset['FQDN'].isna())]

,message,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked


In [19]:
# Counter(uci_dataset['FQDN'].to_list())
uci_dataset[(uci_dataset['Extracted URL'].notna()) & (uci_dataset['FQDN'].notna())]

,message,label,Extracted URL,Message Len,FQDN,Website Size in KB,Website Textual Content Length,Status Code,Parked
2045,"Hi, Mobile no. &lt;#&gt; has added you in th...",ham,www.fullonsms.com,158,www.fullonsms.com,1178.0,0.0,200.0,1.0
4130,"Hi, Mobile no. &lt;#&gt; has added you in th...",ham,www.fullonsms.com,158,www.fullonsms.com,1178.0,0.0,200.0,1.0
4832,URGENT! You have won a 1 week FREE membership ...,spam,www.dbuk.net,155,www.dbuk.net,0.0,0.0,-1.0,0.0
4855,-PLS STOP bootydelious (32/F) is inviting you ...,spam,www.SMS.ac/u/bootydelious,152,www.SMS.ac,1034.0,0.0,200.0,1.0
4859,Are you unique enough? Find out from 30th Augu...,spam,www.areyouunique.co.uk,72,www.areyouunique.co.uk,0.0,0.0,-1.0,0.0
...,...,...,...,...,...,...,...,...,...
5542,XMAS iscoming & ur awarded either £500 CD gift...,spam,www.Ldew.com1win150ppmx3age16subscription,155,www.Ldew.com1win150ppmx3age16subscription.,0.0,0.0,-1.0,0.0
5551,Free entry to the gr8prizes wkly comp 4 a chan...,spam,http//www.gr8prizes.com,159,http.,0.0,0.0,-1.0,0.0
5556,"""For the most sparkling shopping breaks from 4...",spam,www.shortbreaks.org.uk,110,www.shortbreaks.org.uk,0.0,0.0,-1.0,0.0
5558,Txt: CALL to No: 86888 & claim your reward of ...,spam,www.gamb.tv,147,www.gamb.tv,0.0,0.0,-1.0,0.0


In [20]:
print(len(uci_dataset))

5574


In [21]:
#messages with URL
print(len(uci_dataset[(uci_dataset['Extracted URL'].notna())]), len(uci_dataset[(uci_dataset['Extracted URL'].notna())])/len(uci_dataset))

96 0.017222820236813777


In [22]:
#spam messages with URL
print(len(uci_dataset[(uci_dataset['Extracted URL'].notna()) & (uci_dataset['label']=='spam')]), len(uci_dataset[(uci_dataset['Extracted URL'].notna()) & (uci_dataset['label']=='spam')])/len(uci_dataset[uci_dataset['label']=='spam']))

94 0.12583668005354753


In [23]:
#unique FQDN
len(set(uci_dataset[(uci_dataset['FQDN'].notna())]['FQDN']))

54

In [24]:
only_unique_live_websites_data = uci_dataset.drop_duplicates(subset=['FQDN'])
#live websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200)]))

#live websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['label']=='ham')]))

#live websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Status Code']==200) & (only_unique_live_websites_data['label']=='spam')]))

10
1
9


In [25]:
#parked websites full
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1)]))

#parked websites ham
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['label']=='ham')]))

#parked websites spam
print(len(only_unique_live_websites_data[(only_unique_live_websites_data['Parked']==1) & (only_unique_live_websites_data['label']=='spam')]))

6
1
5


In [26]:
uci_dataset.to_csv('../Dataset/Refined_UCI_ML_Repo_Dataset.csv', index=None)